# Задача багатокласової класифікації та регресії в PyTorch

## Частина 1. Задача багатокласової класифікації

### EDA та підготовка ознак

В якості задачі класифікації розглянемо датасет [palmerpenguins](https://github.com/allisonhorst/palmerpenguins).  
Цей датасет містить дані для класифікації пінгвінів трьох видів: **Adélie**, **Gentoo** та **Chinstrap**.  
Класифікація відбувається на основі параметрів: виміри дзьобу, плавців, маси тіла, статі та острову походження.

In [ ]:
from dataclasses import dataclass

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import TargetEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

### Завантаження та попередній перегляд даних

In [ ]:
# Завантажуємо набір даних Palmer Penguins (URL — для Google Colab)
url = 'https://raw.githubusercontent.com/goitacademy/DEEP-LEARNING-FOR-COMPUTER-VISION-AND-NLP/main/data/Module_2_Lecture_2_Class_penguins.csv'
df = pd.read_csv(url)

# Виводимо 5 випадкових рядків для ознайомлення зі структурою
df.sample(5, random_state=42)

### EDA (Exploratory Data Analysis)

In [ ]:
# Базова інформація про набір даних: типи, кількість ненульових значень
df.info()

З колонки `Non-Null Count` бачимо, що лише декілька рядків мають пропущені значення. Видалимо їх.

In [ ]:
# Видаляємо рядки з пропущеними значеннями та скидаємо індекс
df = df.dropna().reset_index(drop=True)

#### Розподіл цільової змінної (`species`)

Хоча розподіл класів не ідеально збалансований, кількість прикладів у кожному класі відносно близька.

> **💡** Коли класи не сильно дисбалансовані, підходящою метрикою є **accuracy**, оскільки вона вимірює частку правильно класифікованих прикладів у загальній їх кількості.

In [ ]:
# Гістограма розподілу цільової змінної species
plt.figure(figsize=(4, 3))
ax = sns.countplot(data=df, x='species')
for i in ax.containers:
    ax.bar_label(i)
    ax.set_xlabel('value')
    ax.set_ylabel('count')

plt.suptitle('Target feature distribution')
plt.tight_layout()
plt.show()

#### Розподіл категоріальної змінної `island`

> **💡** Дисбаланс класів цієї змінної може відображати реальний розподіл даних. Треба бути обережними, щоб не видаляти записи або надлишково їх дискретизувати (oversample), оскільки це може внести зміщення в модель.

In [ ]:
# Гістограма розподілу змінної island
plt.figure(figsize=(4, 3))
ax = sns.countplot(data=df, x='island')
for i in ax.containers:
    ax.bar_label(i)
    ax.set_xlabel('value')
    ax.set_ylabel('count')

plt.suptitle('Island feature distribution')
plt.tight_layout()
plt.show()

#### Попарний розподіл числових ознак

За комбінаціями деяких ознак можна виокремити три кластери (наприклад, `bill_length_mm` та `bill_depth_mm`), але деякі комбінації ознак дають перетини кластерів. Нейронна мережа з нелінійними перетвореннями допоможе розділити ці кластери.

In [ ]:
# Попарний розподіл числових ознак, розфарбований за видом пінгвіна
plt.figure(figsize=(6, 6))
sns.pairplot(data=df, hue='species').fig.suptitle('Numeric features distribution', y=1)
plt.show()

### Підготовка ознак моделі

Залишимо тільки числові змінні для моделювання.

> **💡** В якості «завдання із зірочкою» можете залишити усі ознаки та опрацювати категоріальні змінні, замінивши їх на числові.

In [ ]:
# Залишаємо лише числові ознаки + цільову змінну
features = ['species', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
df = df.loc[:, features]

In [ ]:
# Перетворюємо категоріальну таргетну змінну species в числову
# Adelie → 0, Gentoo → 1, Chinstrap → 2
df.loc[df['species'] == 'Adelie', 'species'] = 0
df.loc[df['species'] == 'Gentoo', 'species'] = 1
df.loc[df['species'] == 'Chinstrap', 'species'] = 2
df = df.apply(pd.to_numeric)

df.head(2)

In [ ]:
# Розділяємо на матрицю ознак X та вектор таргету y (як numpy-масиви)
X = df.drop('species', axis=1).values
y = df['species'].values

Ознаки набору даних мають дуже різний числовий масштаб. Щоб привести їх до одного масштабу, використаємо `StandardScaler`.

In [ ]:
# Стандартизуємо ознаки: середнє = 0, стандартне відхилення = 1
scaler = StandardScaler()
X = scaler.fit_transform(X)

X

In [ ]:
# Розділяємо на тренувальну (67%) та тестову (33%) вибірки
# stratify=y — зберігає пропорції класів у обох вибірках
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.33, stratify=y
)

In [ ]:
# Перетворюємо numpy-масиви в torch.tensor
# X → float (для обчислень у мережі)
# y → long (int64, потрібен для CrossEntropyLoss)
X_train = torch.Tensor(X_train).float()
y_train = torch.Tensor(y_train).long()

X_test = torch.Tensor(X_test).float()
y_test = torch.Tensor(y_test).long()

print(X_train[:1])
print(y_train[:10])

> **💡** `.long()` еквівалентний `.to(torch.int64)` — приведення до цілочисельного типу. Це необхідно, оскільки `CrossEntropyLoss` очікує цілочисельні мітки класів.

---

## Огляд функцій активації

У нейронних мережах **функції активації** визначають результат нейрона. Різні функції слугують різним цілям та впливають на ефективність навчання та узагальнення моделі.

---

### Порогова (ступінчаста) функція

Найпростіша функція активації. Якщо значення перевищує поріг — нейрон активується:

$$z(x) = \begin{cases} 0, & x < threshold \\ 1, & x \geq threshold \end{cases}$$

**Проблеми:**
1. **Недиференційованість** — неможливо обчислити градієнт для зворотного розповсюдження помилки.
2. **Стрибки функції** — різкі зміни вихідних значень навіть при малих змінах входу, через що мережа «застрягає» і не може оптимізувати ваги.

Через ці проблеми в глибокому навчанні **майже не використовується**.

In [ ]:
# Візуалізація порогової (ступінчастої) функції активації
x = np.linspace(-5, 5, 1000)
threshold = np.where(x >= 0, 1, 0)

plt.figure(figsize=(6, 4))
plt.plot(x, threshold, linewidth=2, label='Threshold')
plt.title('Activation Functions')
plt.xlabel('x')
plt.ylabel('Activation')
plt.legend()
plt.grid(True)
plt.show()

---

### Лінійна функція

Зміни вхідного сигналу пропорційно впливають на вихід без нелінійних перетворень:

$$z(x) = x$$

Застосування обмежене **задачами регресії** (передбачення безперервного значення) або як **заключний шар** нейронної мережі для регресії.

In [ ]:
# Візуалізація лінійної функції активації
linear = x

plt.figure(figsize=(6, 4))
plt.plot(x, linear, linewidth=2, label='Linear')
plt.title('Activation Functions')
plt.xlabel('x')
plt.ylabel('Activation')
plt.legend()
plt.grid(True)
plt.show()

---

### Сигмоїдна (логістична) функція

Нелінійна функція, що стискає вхідні значення в діапазон від 0 до 1:

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

Найчастіше застосовується у задачах **бінарної класифікації** для отримання ймовірності приналежності до позитивного класу.

**Недолік:** при великих позитивних або негативних значеннях похідна набуває малих значень → **ефект згасання градієнта** (vanishing gradient).

In [ ]:
# Візуалізація сигмоїдної функції активації
sigmoid = 1 / (1 + np.exp(-x))

plt.figure(figsize=(6, 4))
plt.plot(x, sigmoid, linewidth=2, label='Sigmoid')
plt.title('Activation Functions')
plt.xlabel('x')
plt.ylabel('Activation')
plt.legend()
plt.grid(True)
plt.show()

---

### Ефект згасання градієнта (Vanishing Gradient)

Виникає при навчанні глибоких нейронних мереж: градієнти стають дуже малими при проходженні через шари.

**Механізм:**
1. **Зменшення градієнтів** — при множенні малих значень (< 1) через декілька шарів градієнти експоненційно зменшуються.
2. **Повільне навчання початкових шарів** — ваги майже не оновлюються.
3. **Втрата інформації** — мережа не може вивчати складні закономірності.
4. **Результат** — мережа не здатна ефективно навчитися та узагальнювати.

Особливо проблематичний для **RNN** та глибоких **CNN**.

---

### Гіперболічний тангенс (Tanh)

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} = 2 \cdot \sigma(2x) - 1$$

Вихідні значення лежать в межах **від -1 до 1** → центричність навколо нуля (на відміну від sigmoid з діапазоном [0, 1]).

Має **помітніший градієнт** поблизу нуля порівняно з sigmoid. Корисний для даних із симетричним розподілом навколо нуля.

**Недолік:** проблема згасання градієнта залишається — оновлення ваг відбуватиметься переважно в глибших шарах.

In [ ]:
# Візуалізація функції гіперболічного тангенса
tanh = np.tanh(x)

plt.figure(figsize=(6, 4))
plt.plot(x, tanh, linewidth=2, label='Tanh')
plt.title('Activation Functions')
plt.xlabel('x')
plt.ylabel('Activation')
plt.legend()
plt.grid(True)
plt.show()

---

### Функція активації ReLU

**ReLU** (Rectified Linear Unit) — одна з найпопулярніших функцій активації:

$$f(x) = \max(0, x)$$

**Переваги:**
- **Усуває проблему згасання градієнта** — для значень > 0 градієнт дорівнює 1.
- **Обчислювальна ефективність** — проста операція порівняння.
- **Розрідженість** — активує лише частину нейронів, знижуючи ризик перенавчання.
- **Універсальність** — підходить для будь-яких завдань.

**Недоліки:**
- **Ненормалізованість** — вихід у діапазоні $[0, +\infty)$. Рішення: нормалізація даних перед ReLU.
- **«Мертві нейрони»** — при негативних вагах нейрон завжди виводить 0.

In [ ]:
# Візуалізація ReLU
relu = np.maximum(0, x)

plt.figure(figsize=(6, 4))
plt.plot(x, relu, linewidth=2, label='ReLU')
plt.title('Activation Functions')
plt.xlabel('x')
plt.ylabel('Activation')
plt.legend()
plt.grid(True)
plt.show()

#### Модифікації ReLU

- **Leaky ReLU** — додає невеликий нахил для негативних значень: $f(x) = \max(0.01x, x)$
- **ELU** (Exponential Linear Unit) — має експоненційну залежність негативних значень: $f(x) = \begin{cases} x, & x > 0 \\ \alpha(e^x - 1), & x \leq 0 \end{cases}$

Ці модифікації також запобігають згасанню градієнта та проблемі «мертвих нейронів».

In [ ]:
# Порівняння ReLU, Leaky ReLU та ELU
leaky_relu = np.where(x > 0, x, 0.01 * x)
alpha = 1.0
elu = np.where(x > 0, x, alpha * (np.exp(x) - 1))

plt.figure(figsize=(6, 4))
plt.plot(x, relu, linewidth=2, label='ReLU')
plt.plot(x, leaky_relu, linewidth=2, label='Leaky ReLU')
plt.plot(x, elu, linewidth=2, label='ELU')
plt.title('Activation Functions')
plt.xlabel('x')
plt.ylabel('Activation')
plt.legend()
plt.grid(True)
plt.show()

---

## Частина 2. Моделювання задачі класифікації

### Підготовка моделі

Створимо нейронну мережу з:
- **Лінійний шар** (4 → 20 нейронів)
- **ReLU** — нелінійна активація
- **Лінійний шар** (20 → 3 нейрони, за кількістю класів)
- **Softmax** — перетворення логітів у ймовірності

### Функція Softmax

Перетворює вектор логітів $z = [z_1, z_2, \ldots, z_N]$ у вектор ймовірностей:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{N} e^{z_j}}$$

**Властивості:**
- **Нормалізація** — сума ймовірностей дорівнює 1.
- **Порівнянність** — легко порівнювати ймовірності різних класів.
- **Диференційованість** — працює з градієнтним спуском.

**Недоліки:**
- Чутливість до великих логітів (малі градієнти для інших класів).
- Вхідні логіти варто масштабувати для числової стабільності.

In [ ]:
# Клас нейронної мережі для багатокласової класифікації
class LinearModel(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=20, out_dim=3):
        super().__init__()

        self.features = torch.nn.Sequential(
            nn.Linear(in_dim, hidden_dim),   # Перший лінійний шар: 4 → 20
            torch.nn.ReLU(),                 # Нелінійна активація
            nn.Linear(hidden_dim, out_dim),  # Другий лінійний шар: 20 → 3
            nn.Softmax(dim=1)                # Перетворення у ймовірності
        )

    def forward(self, x):
        output = self.features(x)
        return output

In [ ]:
# Ініціалізація моделі:
# in_dim = кількість ознак, hidden_dim = 20, out_dim = 3 класи
model = LinearModel(X_train.shape[1], 20, 3)

### Функція втрат Cross-Entropy Loss

Для задачі багатокласової класифікації з $N$ класами:

$$\text{Loss} = -\sum_{i=1}^{N} y_i \log(p_i)$$

де:
- $y_i$ — істинна ймовірність класу $i$ (1 для істинного класу, 0 для інших; one-hot кодування),
- $p_i$ — передбачена ймовірність класу $i$ (після Softmax).

**Переваги:**
- **Чутливість до ймовірностей** — враховує не лише правильність, а й впевненість передбачення.
- **«Штрафування»** — сильно карає модель за високі ймовірності для неправильних класів.

**Недоліки:**
- Чутливість до впевнених неправильних передбачень.
- Потреба у нормалізації передбачень (через Softmax).

> **👉🏻** «Штрафування» означає, що функція втрат присвоює **високе значення** прогнозам, далеким від справжніх, і **низьке** — близьким. Це спонукає модель вчитися робити точніші прогнози.

In [ ]:
# Функція втрат: Cross-Entropy Loss
criterion = nn.CrossEntropyLoss()

# Оптимізатор: Stochastic Gradient Descent з learning rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Кількість епох навчання
num_epoch = 400

# Списки для збереження результатів тренування
train_loss = []
test_loss = []
train_accs = []
test_accs = []

### Цикл навчання та тестування

**Тренування (train):**
1. `model.train()` — переводимо в режим навчання (активує Dropout, BatchNorm тощо).
2. Forward pass — отримуємо передбачення.
3. Обчислюємо loss.
4. Backward pass + оптимізація — оновлюємо ваги.
5. Обчислюємо accuracy.

**Тестування (eval):**
1. `model.eval()` — режим валідації (градієнти не обчислюються).
2. `torch.no_grad()` — гарантія відсутності розрахунку градієнтів.
3. Forward pass + loss + accuracy (без backward pass).

In [ ]:
for epoch in range(num_epoch):

    # === Тренування ===
    model.train()

    outputs = model(X_train)                              # Forward pass

    loss = criterion(outputs, y_train)                    # Обчислення loss
    train_loss.append(loss.cpu().detach().numpy())

    optimizer.zero_grad()                                 # Обнулення градієнтів
    loss.backward()                                      # Backward pass
    optimizer.step()                                      # Оновлення ваг

    acc = 100 * torch.sum(y_train == torch.max(outputs.data, 1)[1]).double() / len(y_train)
    train_accs.append(acc)

    if (epoch + 1) % 50 == 0:
        print('Epoch [%d/%d] Loss: %.4f   Acc: %.4f'
              % (epoch + 1, num_epoch, loss.item(), acc.item()))

    # === Тестування ===
    model.eval()
    with torch.no_grad():
        outputs = model(X_test)

        loss = criterion(outputs, y_test)
        test_loss.append(loss.cpu().detach().numpy())

        acc = 100 * torch.sum(y_test == torch.max(outputs.data, 1)[1]).double() / len(y_test)
        test_accs.append(acc)

### Аналіз результатів класифікації

In [ ]:
# Графік функції втрат: Training vs Validation
plt.figure(figsize=(4, 3))
plt.plot(train_loss, label='Train')
plt.plot(test_loss, label='Validation')
plt.legend(loc='best')
plt.xlabel('Epochs')
plt.ylabel('Cross-Entropy Loss')
plt.title('Training vs Validation Loss')
plt.show()

# Графік точності: Training vs Validation
plt.figure(figsize=(4, 3))
plt.plot(train_accs, label='Train')
plt.plot(test_accs, label='Validation')
plt.legend(loc='best')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Metric')
plt.show()

**Висновки:**
- Loss-функція рівномірно спадає. Графіки train/validation майже ідентичні → немає пере- чи недонавчання.
- Accuracy досягає ~90%.
- Для цієї задачі достатньо невеликої мережі з двох шарів. Модель **зійшлась** (converged) за 400 епох.

> **👉🏼** Коли говорять, що модель «зійшлась», мають на увазі, що процес навчання досяг стабільного стану — значення loss перестали значно зменшуватися, і модель перестала суттєво покращувати передбачення.

---

## Частина 3. Задача регресії

В якості прикладу задачі регресії використаємо набір даних [BigMart Sales](https://www.analyticsvidhya.com/datahack/contest/practice-problem-big-mart-sales-iii/).

### Підготовка даних

Відтворимо підготовку даних, як вона була зроблена на курсі «Machine Learning: Fundamentals and Applications».

In [ ]:
# Завантажуємо набір даних BigMart Sales (URL — для Google Colab)
url_bigmart = 'https://raw.githubusercontent.com/goitacademy/DEEP-LEARNING-FOR-COMPUTER-VISION-AND-NLP/main/data/Module_2_Lecture_2_Class_bigmart_data.csv'
data = pd.read_csv(url_bigmart)
data.head(3)

In [ ]:
# Відтворюємо попередню обробку даних з курсу ML

data['Outlet_Establishment_Year'] = 2013 - data['Outlet_Establishment_Year']
data['Item_Visibility'] = (
    data['Item_Visibility'].mask(data['Item_Visibility'].eq(0), np.nan)
)

data['Item_Visibility_Avg'] = (
    data.groupby(['Item_Type', 'Outlet_Type'])['Item_Visibility'].transform('mean')
)

data['Item_Visibility'] = (
    data['Item_Visibility'].fillna(data['Item_Visibility_Avg'])
)

data['Item_Visibility_Ratio'] = (
    data['Item_Visibility'] / data['Item_Visibility_Avg']
)

data['Item_Fat_Content'] = data['Item_Fat_Content'].replace({
    'low fat': 'Low Fat',
    'LF': 'Low Fat',
    'reg': 'Regular'
})

data['Item_Identifier_Type'] = data['Item_Identifier'].str[:2]

data_num = data.select_dtypes(include=np.number)
data_cat = data.select_dtypes(include='object')

In [ ]:
# Розділення на train/test та обробка пропусків і категоріальних ознак

X_train_num, X_test_num, X_train_cat, X_test_cat, y_train, y_test = (
    train_test_split(
        data_num.drop(['Item_Outlet_Sales', 'Item_Visibility_Avg'], axis=1).values,
        data_cat.drop('Item_Identifier', axis=1).values,
        data['Item_Outlet_Sales'].values,
        test_size=0.2,
        random_state=42
    )
)

# Імпутація числових ознак (середнім)
num_imputer = SimpleImputer().set_output(transform='pandas')
X_train_num = num_imputer.fit_transform(X_train_num)
X_test_num = num_imputer.transform(X_test_num)

# Імпутація категоріальних ознак (модою)
cat_imputer = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')
X_train_cat = cat_imputer.fit_transform(X_train_cat)
X_test_cat = cat_imputer.transform(X_test_cat)

# Target Encoding категоріальних ознак
enc_auto = TargetEncoder(random_state=42).set_output(transform='pandas')
X_train_cat = enc_auto.fit_transform(X_train_cat, y_train)
X_test_cat = enc_auto.transform(X_test_cat)

# Об'єднання числових та категоріальних ознак
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

### Представлення даних в PyTorch: Dataset та DataLoader

PyTorch передбачає можливість опрацьовувати дані **батчами**.

> **💡 Batch processing** — метод обробки даних, за якого навчання виконується на підмножині даних (батч/міні-батч), замість обробки всього набору за один раз. Це дозволяє ефективніше використовувати ресурси та стабілізувати навчання.

---

#### Клас `Dataset`

Абстракція для представлення набору даних. Визначає, як отримувати зразки та мітки.

Необхідно реалізувати два методи:
- `__len__()` — кількість зразків у наборі.
- `__getitem__(idx)` — повертає `i`-й зразок та відповідну мітку.

In [ ]:
class BigmartDataset(Dataset):
    def __init__(self, X, y, scale=True):
        self.X = X.values  # DataFrame → NumPy array
        self.y = y

        if scale:
            sc = StandardScaler()
            self.X = sc.fit_transform(self.X)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        X = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return X, y

In [ ]:
# Створюємо об'єкти Dataset
train_dataset = BigmartDataset(X_train, y_train)
test_dataset = BigmartDataset(X_test, y_test)

#### Клас `DataLoader`

Інструмент для ітерації по набору даних у вигляді батчів:
- `batch_size` — розмір батчу.
- `shuffle=True` — перемішування даних перед кожною епохою.
- `num_workers` — кількість потоків для паралельного завантаження.

In [ ]:
# Створюємо DataLoader для батчевої обробки даних
train_dataloader = DataLoader(train_dataset, batch_size=200, num_workers=4)
test_dataloader = DataLoader(test_dataset, batch_size=200, num_workers=4)

In [ ]:
# Перевіряємо, що DataLoader коректно генерує батч
next(iter(train_dataloader))

### Моделювання задачі регресії

Ця задача складніша, тому створимо мережу з **більшою кількістю шарів** та нейронів.

> **💡** Зверніть увагу: останнім є **лінійний шар** (без функції активації), оскільки для регресії ми передбачаємо фактичне значення цільової змінної.

In [ ]:
class LinearModel(torch.nn.Module):
    def __init__(self, in_dim, out_dim=1):
        super().__init__()

        self.features = torch.nn.Sequential(
            nn.Linear(in_dim, 256),    # 1-й шар: вхід → 256
            torch.nn.ReLU(),
            nn.Linear(256, 128),       # 2-й шар: 256 → 128
            torch.nn.ReLU(),
            nn.Linear(128, 64),        # 3-й шар: 128 → 64
            torch.nn.ReLU(),
            nn.Linear(64, out_dim),    # Вихідний шар: 64 → 1 (без активації!)
        )

    def forward(self, x):
        output = self.features(x)
        return output

In [ ]:
# Ініціалізація моделі (вихідний шар — 1 нейрон для регресії)
model = LinearModel(in_dim=X_train.shape[1], out_dim=1)

# Функція втрат: Mean Squared Error
criterion = nn.MSELoss()

# Оптимізатор: Adam з learning rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_losses = []
train_rmses = []
test_losses = []
test_rmses = []

### Цикл навчання з батчами

> **💡** Зверніть увагу на **цикл у циклі**: зовнішній — по епохах, внутрішній — по батчах. Одна епоха проходить, коли модель побачить усі тренувальні дані, тобто проітерується усіма батчами.

In [ ]:
num_epochs = 100

for epoch in range(num_epochs):

    # === Train step ===
    model.train()
    y_pred_train = []

    for data_batch in train_dataloader:
        inputs, targets = data_batch
        inputs, targets = inputs.float(), targets.float()
        targets = targets.reshape((targets.shape[0], 1))

        outputs = model(inputs)                 # Forward pass
        loss = criterion(outputs, targets)       # Обчислення loss

        optimizer.zero_grad()                    # Обнулення градієнтів
        loss.backward()                          # Backward pass
        optimizer.step()                         # Оновлення ваг

        y_pred_train.extend(outputs.cpu().detach().numpy())

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, RMSE: {RMSE(y_train, y_pred_train)}')
    train_rmses.append(RMSE(y_train, y_pred_train))
    train_losses.append(loss.cpu().detach().numpy())

    # === Eval step ===
    model.eval()
    y_pred_test = []

    with torch.no_grad():
        for data_batch in test_dataloader:
            inputs, targets = data_batch
            inputs, targets = inputs.float(), targets.float()
            targets = targets.reshape((targets.shape[0], 1))

            outputs = model(inputs)              # Forward pass (без backward!)
            loss = criterion(outputs, targets)

            y_pred_test.extend(outputs.cpu().detach().numpy())

        test_rmses.append(RMSE(y_test, y_pred_test))
        test_losses.append(loss.cpu().detach().numpy())

### Аналіз результатів регресії

In [ ]:
# Графік функції втрат (MAE)
plt.figure(figsize=(4, 3))
plt.plot(train_losses, label='Train')
plt.plot(test_losses, label='Validation')
plt.legend(loc='best')
plt.xlabel('Epochs')
plt.ylabel('MAE')
plt.title('Training vs Validation Loss')
plt.show()

# Графік метрики RMSE
plt.figure(figsize=(4, 3))
plt.plot(train_rmses, label='Train')
plt.plot(test_rmses, label='Validation')
plt.legend(loc='best')
plt.xlabel('Epochs')
plt.ylabel('RMSE')
plt.title('Training vs Validation Metric - RMSE')
plt.show()

**Висновки:**
- Функція втрат **рівномірно спадає** — модель поступово краще узгоджує прогнози з реальними значеннями.
- **RMSE зменшується** — очікуваний результат для задачі регресії.
- Навчання пройшло успішно — модель демонструє покращену здатність до прогнозування.